In [36]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
╔══════════════════════════════════════════════════════════════════════════════╗
║                             SLR Manuscript                                   ║
║    Survival Analysis × Granger Causality × Demand Forecasting                ║
║      figures publication-ready · Matplotlib + Seaborn                        ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""

import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.gridspec import GridSpec
import seaborn as sns
from collections import Counter, defaultdict
from itertools import combinations
from scipy.stats import chi2_contingency, fisher_exact
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.manifold import TSNE
import networkx as nx

warnings.filterwarnings("ignore")

In [2]:
# ═══════════════════════════════════════════════════════════════════
#  STYLE CONFIGURATION — Publication-grade academic palette
# ═══════════════════════════════════════════════════════════════════
PALETTE = {
    "C1": "#2171B5", "C2": "#238B45", "C3": "#D95F02",
    "C4": "#7570B3", "C5": "#E7298A", "OTHER": "#999999",
    "bg": "#FAFAFA", "grid": "#ECECEC", "text": "#2D2D2D",
    "accent": "#1A3A6B", "red": "#C62828", "amber": "#F57F17",
    "navy": "#0D3B66", "light_blue": "#E3F2FD",
}
CLUSTER_NAMES = {
    "C1": "Survival methods", "C2": "Customer analytics",
    "C3": "Granger causality", "C4": "Causal inference",
    "C5": "Demand forecasting", "OTHER": "Other"
}

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 11,
    "axes.titlesize": 14, "axes.titleweight": "bold",
    "axes.labelsize": 12, "axes.facecolor": PALETTE["bg"],
    "figure.facecolor": "white", "figure.dpi": 150,
    "savefig.bbox": "tight", "savefig.dpi": 200,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3,
})

OUT = "Output2"
os.makedirs(OUT, exist_ok=True)

def savefig(fig, name):
    fig.savefig(f"{OUT}/{name}.png", facecolor="white", edgecolor="none")
    plt.close(fig)
    print(f"  ✓ {name}.png")

In [3]:
# ═══════════════════════════════════════════════════════════════════
#  DATA LOADING & PREPARATION 
# ═══════════════════════════════════════════════════════════════════
print("═" * 60)
print("  Data Loaded ...")
print("═" * 60)

CSV = "combined_bibliometric_data.csv"
df = pd.read_csv(CSV, dtype=str).fillna("")
df["year"]         = pd.to_numeric(df["year"],         errors="coerce")
df["author_count"] = pd.to_numeric(df["author_count"], errors="coerce").fillna(0).astype(int)

def clean(text):
    return re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", str(text))).strip()

df["abstract_clean"] = df["abstract"].apply(clean)
df["title_clean"]    = df["title"].str.strip()
df["full_text"]      = (df["title_clean"].str.lower() + " " +
                        df["abstract_clean"].str.lower() + " " +
                        df["subjects"].str.lower())

════════════════════════════════════════════════════════════
  Data Loaded ...
════════════════════════════════════════════════════════════


In [4]:
# ═══════════════════════════════════════════════════════════════════════════
#  ALGORITHMIC RELEVANCE SCORING
# ═══════════════════════════════════════════════════════════════════════════

def section_header(num, title):
    print(f"\n{'=' * 72}")
    print(f"  STEP {num} — {title}")
    print(f"{'=' * 72}\n")


def save_table(df, name):
    path = os.path.join(TABLE_DIR, f"{name}.csv")
    df.to_csv(path, index=True)
    print(f"  ✓ Table saved: {path}")

OUTPUT_DIR = "/Users/juniormukenze/Documents/00 Personal Research Publication/Thesis/Systematic Review Survival Analysis/from_Scratch_SLR_2026/Output2"
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")


In [5]:
# ── Cluster assignment — version enrichie (critères STEP 3) ──────
CLUSTER_KEYWORDS = {
    "C1": [
        "survival analysis", "cox proportional", "kaplan-meier", "kaplan meier",
        "time-to-event", "time to event", "hazard ratio", "hazard function",
        "censored", "censoring", "competing risk", "deepsurv", "deephit",
        "accelerated failure", "log-rank", "frailty model", "proportional hazards",
        "survival curve", "survival function", "survival model", "survival prediction",
        "nelson-aalen", "cumulative incidence"
    ],
    "C2": [
        "customer churn", "churn prediction", "customer lifetime", "clv ", "cltv",
        "retention", "subscriber", "attrition", "loyalty",
        "rfm", "e-commerce churn", "telecom churn", "lifetime value"
    ],
    "C3": [
        "granger causality", "granger-cause", "granger cause",
        "vector autoregression", "var model", "predictive causality",
        "temporal granger", "spectral granger"
    ],
    "C4": [
        "causal inference", "counterfactual", "treatment effect",
        "propensity score", "directed acyclic graph", "dag ", " dag,",
        "structural causal", "scm ", "confounder", "confounding",
        "double robust", "doubly robust", "tmle", "dml ",
        "instrumental variable", "regression discontinuity",
        "difference-in-difference", "potential outcomes",
        "average treatment effect", "heterogeneous treatment",
        "causal discovery", "causal effect", "do-calculus"
    ],
    "C5": [
        "demand forecasting", "demand prediction", "energy demand",
        "supply chain", "sales forecast", "arima", "forecast accuracy",
        "load forecasting", "electricity demand", "power demand", "inventory"
    ]
}

NEGATIVE_KEYWORDS = [
    "gene expression", "genome-wide", "protein structure",
    "surgical technique", "clinical trial phase",
    "drug discovery", "molecular docking",
    "image segmentation only", "object detection only"
]

In [6]:
def assign_cluster(row):
    title    = str(row["title_clean"]).lower()
    abstract = str(row["abstract_clean"]).lower()
    subjects = str(row["subjects"]).lower()

    scores    = {c: 0 for c in CLUSTER_KEYWORDS}
    activated = set()

    for cluster, keywords in CLUSTER_KEYWORDS.items():
        for kw in keywords:
            if kw in title:    scores[cluster] += 2
            if kw in abstract: scores[cluster] += 1
            if kw in subjects: scores[cluster] += 1
        if scores[cluster] > 0:
            activated.add(cluster)

    full = title + " " + abstract
    for neg_kw in NEGATIVE_KEYWORDS:
        if neg_kw in full:
            for c in scores:
                scores[c] -= 2

    if len(activated) >= 2:
        for c in activated:
            scores[c] += 3

    best = max(scores, key=scores.get)
    if scores[best] <= 0:
        if any(w in full for w in ["survival", "hazard", "censored"]):  return "C1"
        if any(w in full for w in ["churn", "customer"]):               return "C2"
        if "granger" in full:                                           return "C3"
        if any(w in full for w in ["causal", "counterfactual"]):        return "C4"
        if any(w in full for w in ["forecast", "demand"]):              return "C5"
        return "OTHER"
    return best

df["cluster"] = df.apply(assign_cluster, axis=1)

In [7]:
df.shape

(1877, 17)

In [8]:
# ── Relevance scoring  ─
# Use of 3 transversal thematic categories,
# distincts  5 clusters
KEYWORDS = {
    "method_A_survival": [
        "survival analysis", "cox proportional", "kaplan-meier", "kaplan meier",
        "time-to-event", "time to event", "hazard ratio", "hazard function",
        "censored", "censoring", "competing risk", "deepsurv", "deephit",
        "accelerated failure", "log-rank", "frailty model", "proportional hazards",
        "survival curve", "survival function", "survival model", "survival prediction",
        "nelson-aalen", "cumulative incidence"
    ],
    "method_B_causality": [
        "granger causality", "granger-cause", "granger cause",
        "causal inference", "counterfactual", "treatment effect",
        "propensity score", "directed acyclic graph", "dag ", " dag,",
        "structural causal", "scm ", "confounder", "confounding",
        "double robust", "doubly robust", "tmle", "dml ",
        "instrumental variable", "regression discontinuity",
        "difference-in-difference", "potential outcomes",
        "average treatment effect", "heterogeneous treatment",
        "causal discovery", "causal effect", "do-calculus"
    ],
    "application_domain": [
        "demand forecasting", "demand prediction", "customer churn",
        "churn prediction", "customer lifetime", "clv ", "cltv",
        "sales forecast", "supply chain", "energy demand",
        "retention", "subscriber", "attrition", "loyalty",
        "rfm", "e-commerce churn", "telecom churn"
    ]
}

In [9]:
def compute_relevance_score(row):
    """Score pondéré identique à STEP 3 :
       titre ×2 / abstract ×1 / subjects ×1
       pénalité −2 par mot hors-scope
       bonus +3 si ≥ 2 catégories activées
       Retourne les 4 colonnes produites par STEP 3.
    """
    title    = row["title_clean"].lower()
    abstract = row["abstract_clean"].lower()
    subjects = row["subjects"].lower()

    score             = 0
    matched_categories = set()
    matched_keywords  = []

    for category, keywords in KEYWORDS.items():
        cat_score = 0
        for kw in keywords:
            if kw in title:
                cat_score += 2
                matched_keywords.append(f"[T]{kw}")
            if kw in abstract:
                cat_score += 1
                matched_keywords.append(f"[A]{kw}")
            if kw in subjects:
                cat_score += 1
                matched_keywords.append(f"[S]{kw}")
        if cat_score > 0:
            matched_categories.add(category)
        score += cat_score

    full = title + " " + abstract
    for neg_kw in NEGATIVE_KEYWORDS:
        if neg_kw in full:
            score -= 2

    if len(matched_categories) >= 2:
        score += 3

    return pd.Series({
        "relevance_score":          score,
        "matched_categories":       "|".join(sorted(matched_categories)),
        "n_categories":             len(matched_categories),
        "matched_keywords_sample":  "; ".join(matched_keywords[:8])
    })

print("  Computing relevance scores...")
scores_df = df.apply(compute_relevance_score, axis=1)
df = pd.concat([df, scores_df], axis=1)
df["relevance_score"] = df["relevance_score"].astype(int)

THRESHOLD   = 3
df_filtered = df[df["relevance_score"] >= THRESHOLD].copy()

n_before = len(df)
n_after  = len(df_filtered)
print(f"  ► Threshold applied : score ≥ {THRESHOLD}")
print(f"  ► Before filtering  : {n_before} articles")
print(f"  ► After filtering   : {n_after} articles  ({n_after/n_before*100:.1f}% retained)")


  Computing relevance scores...
  ► Threshold applied : score ≥ 3
  ► Before filtering  : 1877 articles
  ► After filtering   : 612 articles  (32.6% retained)


In [10]:
# ── Method & Domain categorization ───────────────────────────────
def cat_method(t):
    t = str(t).lower()
    if any(w in t for w in ["survival analysis","cox proportional","kaplan-meier","hazard","time-to-event","censored"]): return "Survival"
    if any(w in t for w in ["granger causality","granger cause","var model"]):                                           return "Granger"
    if any(w in t for w in ["causal inference","counterfactual","treatment effect","propensity score","causal effect"]): return "Causal Inference"
    if any(w in t for w in ["deep learning","neural network","lstm","transformer","xgboost","random forest","machine learning","gradient boosting"]): return "ML/DL"
    return "Other"

def cat_domain(t):
    t = str(t).lower()
    if any(w in t for w in ["cancer","clinical","patient","medical","disease","mortality","tumor","prognosis"]): return "Medical"
    if any(w in t for w in ["churn","customer","retail","e-commerce","subscriber","loyalty","telecom"]):         return "Business"
    if any(w in t for w in ["energy","electricity","power demand","solar","wind","grid"]):                       return "Energy"
    if any(w in t for w in ["financial","stock","market","gdp","economic","monetary","exchange rate"]):          return "Finance"
    if any(w in t for w in ["forecast","demand","supply chain","inventory"]):                                    return "Forecasting"
    return "Other"

df_filtered["method_cat"] = df_filtered["full_text"].apply(cat_method)
df_filtered["domain_cat"] = df_filtered["full_text"].apply(cat_domain)

print(f"  Clusters     : {df_filtered['cluster'].value_counts().to_dict()}")
print(f"  Methods      : {df_filtered['method_cat'].value_counts().to_dict()}")
print(f"  Domains      : {df_filtered['domain_cat'].value_counts().to_dict()}")
print(f"  Prêt pour visualisations.\n")

  Clusters     : {'C1': 306, 'C2': 154, 'C4': 82, 'C5': 45, 'C3': 25}
  Methods      : {'Survival': 294, 'Other': 130, 'ML/DL': 82, 'Causal Inference': 78, 'Granger': 28}
  Domains      : {'Medical': 210, 'Business': 171, 'Other': 139, 'Forecasting': 51, 'Finance': 21, 'Energy': 20}
  Prêt pour visualisations.



In [11]:
# ─── Score distribution ──────────────────────────────────────────────
print("\n  Relevance score distribution:")
print(f"  {'Score':>8} | {'Articles':>10} | {'Cumulative %':>12}")
print(f"  {'─' * 8}-+-{'─' * 10}-+-{'─' * 12}")

score_dist = scores_df["relevance_score"].value_counts().sort_index()
total = len(scores_df)
cumul = 0

for score_val, count in score_dist.items():
    cumul += count
    print(f"  {score_val:>8} | {count:>10} | {cumul / total * 100:>11.1f}%")


  Relevance score distribution:
     Score |   Articles | Cumulative %
  ────────-+-──────────-+-────────────
        -2 |         16 |         0.9%
        -1 |          3 |         1.0%
         0 |        603 |        33.1%
         1 |        119 |        39.5%
         2 |        524 |        67.4%
         3 |        138 |        74.7%
         4 |        209 |        85.9%
         5 |         54 |        88.8%
         6 |         82 |        93.1%
         7 |         43 |        95.4%
         8 |         29 |        97.0%
         9 |         24 |        98.2%
        10 |          9 |        98.7%
        11 |         10 |        99.3%
        12 |          5 |        99.5%
        13 |          2 |        99.6%
        14 |          3 |        99.8%
        15 |          1 |        99.8%
        18 |          2 |        99.9%
        19 |          1 |       100.0%


In [12]:
# ─── Visualization of score distribution ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1 = axes[0]

min_score = int(scores_df["relevance_score"].min())
max_score = int(scores_df["relevance_score"].max())
bins = range(min_score, max_score + 2)

ax1.hist(
    scores_df["relevance_score"],
    bins=bins,
    color="#4A90D9",
    edgecolor="white",
    alpha=0.85
)

THRESHOLD = 3

ax1.axvline(
    x=THRESHOLD,
    color="#D32F2F",
    linestyle="--",
    linewidth=2,
    label=f"Threshold = {THRESHOLD}"
)

ax1.set_xlabel("Relevance score")
ax1.set_ylabel("Number of articles")
ax1.set_title("Distribution of relevance scores")
ax1.legend()

# Matched categories
ax2 = axes[1]

cat_counts = scores_df["n_categories"].value_counts().sort_index()

colors = {
    0: "#CCCCCC",
    1: "#8BC34A",
    2: "#FF9800",
    3: "#D32F2F"
}

bar_colors = [colors.get(int(i), "#D32F2F") for i in cat_counts.index]

ax2.bar(
    cat_counts.index,
    cat_counts.values,
    color=bar_colors,
    edgecolor="white"
)

ax2.set_xlabel("Number of matched categories")
ax2.set_ylabel("Number of articles")
ax2.set_title("Cross-domain coverage")
ax2.set_xticks(cat_counts.index)

for i, v in cat_counts.items():
    ax2.text(
        i,
        v + max(cat_counts.values) * 0.02,
        str(v),
        ha="center",
        fontsize=10,
        fontweight="bold"
    )

fig.suptitle(
    " Algorithmic Relevance Scoring",
    fontsize=14,
    fontweight="bold",
    y=1.02
)

plt.tight_layout()
savefig(fig, "fig_01")


  ✓ fig_01.png


In [13]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║        FIGURE 2 —   PRISMA FLOW DIAGRAM                          ║
# ╚══════════════════════════════════════════════════════════════════╝
print("═" * 60)
print("  FIGURE 2 — PRISMA FLOW DIAGRAM")
print("═" * 60)

fig, ax = plt.subplots(figsize=(12, 10))
ax.set_xlim(0, 12); ax.set_ylim(0, 12)
ax.axis("off")

# Boxes
boxes_data = [
    (4.5, 10.5, 4.5, 1.0, f"IDENTIFICATION\n1,877 records\n(CrossRef: 1,232 · arXiv: 332 · PubMed: 313)", "#E3F2FD", "#1565C0"),
    (4.5, 8.5, 4.5, 0.8, f"After deduplication\n1,568 records (DOI matching)", "#E8F5E9", "#2E7D32"),
    (4.5, 6.5, 4.5, 0.8, f"After type filtering\n1,393 records", "#FFF3E0", "#E65100"),
    (4.5, 4.5, 4.5, 0.8, f"After relevance scoring (≥ 3)\n{len(df_filtered)} articles", "#F3E5F5", "#6A1B9A"),
    (4.5, 2.5, 4.5, 1.0, f"FINAL ANALYTICAL CORPUS\n{len(df_filtered)} articles", "#FFEBEE", "#C62828"),
]

for x, y, w, h, text, fc, ec in boxes_data:
    rect = FancyBboxPatch((x - w/2, y - h/2), w, h, boxstyle="round,pad=0.15",
                           facecolor=fc, edgecolor=ec, linewidth=2.5)
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=10.5, fontweight="bold",
            multialignment="center", color=PALETTE["text"])

# Arrows
for i in range(len(boxes_data) - 1):
    y_start = boxes_data[i][1] - boxes_data[i][3]/2 - 0.05
    y_end = boxes_data[i+1][1] + boxes_data[i+1][3]/2 + 0.05
    ax.annotate("", xy=(4.5, y_end), xytext=(4.5, y_start),
                arrowprops=dict(arrowstyle="-|>", color="#444", lw=2.5, mutation_scale=18))

# Exclusion boxes (right side)
excl = [
    (9.5, 8.5, "Duplicates\nremoved: 309", "#F5F5F5"),
    (9.5, 6.5, f"Types excluded:\n−175 records", "#FFF8E1"),
    (9.5, 4.5, f"Score < 3:\n−{781} records", "#FBE9E7"),
    #(9.5, 4.5, f"Score < 3:\n−{1702 - len(df_filtered)} records", "#FBE9E7"),
]
for x, y, text, fc in excl:
    rect = FancyBboxPatch((x - 1.2, y - 0.35), 2.4, 0.7, boxstyle="round,pad=0.1",
                           facecolor=fc, edgecolor="#CCC", linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=9, color="#666", multialignment="center")
    ax.annotate("", xy=(x - 1.2, y), xytext=(4.5 + 2.25, y),
                arrowprops=dict(arrowstyle="-|>", color="#CCC", lw=1.5, linestyle="--"))

ax.set_title("Figure — PRISMA-like Selection Pipeline", fontsize=16, pad=20,
             fontweight="bold", color=PALETTE["navy"])

savefig(fig,"fig_02")


════════════════════════════════════════════════════════════
  FIGURE 2 — PRISMA FLOW DIAGRAM
════════════════════════════════════════════════════════════
  ✓ fig_02.png


In [14]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FIGURE 3 — TEMPORAL EVOLUTION                                   ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 3 — Temporal Evolution")

yc = pd.crosstab(df_filtered["year"], df_filtered["cluster"])
yc = yc.reindex(columns=["C1","C2","C3","C4","C5","OTHER"], fill_value=0)
yc = yc.loc[yc.index.dropna()]
yc.index = yc.index.astype(int)
yc = yc.loc[2020:2026]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1, 1.2]})

# Left: total bar chart
totals = yc.sum(axis=1)
bars = ax1.bar(totals.index, totals.values, color=PALETTE["C1"], edgecolor="white", width=0.65, zorder=3)
for bar, val in zip(bars, totals.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
             str(val), ha="center", fontweight="bold", fontsize=11, color=PALETTE["text"])
ax1.set_xlabel("Year"); ax1.set_ylabel("Number of articles")
ax1.set_title("a) Overall publication volume")
ax1.set_xticks(totals.index)

# Growth annotation
if len(totals) >= 6:
    growth = (totals.iloc[-2] / totals.iloc[0] - 1) * 100
    ax1.annotate(f"+{growth:.0f}%\n(2020→2025)", xy=(2025, totals.loc[2025]),
                 xytext=(2022.5, totals.loc[2025] + 25),
                 arrowprops=dict(arrowstyle="->", color=PALETTE["red"], lw=2),
                 fontsize=11, color=PALETTE["red"], fontweight="bold")

# Right: stacked area
clusters = ["C1","C2","C3","C4","C5"]
for c in clusters:
    ax2.fill_between(yc.index, 0, 0, alpha=0)  # init
bottom = np.zeros(len(yc))
for c in clusters:
    vals = yc[c].values.astype(float)
    ax2.bar(yc.index, vals, bottom=bottom, label=f"{c} — {CLUSTER_NAMES[c]}",
            color=PALETTE[c], edgecolor="white", width=0.65, zorder=3)
    bottom += vals
ax2.set_xlabel("Year"); ax2.set_ylabel("Number of articles")
ax2.set_title("b) Distribution by thematic cluster")
ax2.legend(fontsize=9, loc="upper left", framealpha=0.9)
ax2.set_xticks(yc.index)

fig.suptitle("Figure 2 — Temporal Evolution of the Corpus (2020–2026 March)",
             fontsize=15, fontweight="bold", color=PALETTE["navy"], y=1.02)
plt.tight_layout()
savefig(fig, "fig_03")

  FIGURE 3 — Temporal Evolution
  ✓ fig_03.png


In [15]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FIGURE 4 — CLUSTER TREND LINES (Section 3.1.2)                 ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 4 — Cluster Trend Lines")

fig, ax = plt.subplots(figsize=(12, 6))
for c in clusters:
    ax.plot(yc.index, yc[c], marker="o", linewidth=2.8, markersize=8,
            label=f"{c} — {CLUSTER_NAMES[c]}", color=PALETTE[c], zorder=3)
    # End label
    ax.text(yc.index[-1] + 0.15, yc[c].iloc[-1], c, fontsize=9,
            fontweight="bold", color=PALETTE[c], va="center")

ax.set_xlabel("Year"); ax.set_ylabel("Number of articles")
ax.set_title("Figure 3 — Cluster Growth Trends (2020–2026)",
             fontsize=14, fontweight="bold", color=PALETTE["navy"])
ax.legend(fontsize=9, loc="upper left")
ax.set_xticks(yc.index)
plt.tight_layout()
savefig(fig, "fig_04")

  FIGURE 4 — Cluster Trend Lines
  ✓ fig_04.png


In [16]:

# ╔══════════════════════════════════════════════════════════════════╗
# ║  FIGURE 5 — JOURNAL / PUBLISHER DISTRIBUTION (Section 3.1.2-3)  ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 5 — Journal & Publisher Distribution")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Left: Top journals
j_counts = df_filtered[df_filtered["journal"].str.strip() != ""]["journal"].value_counts().head(12)
y_pos = range(len(j_counts))
ax1.barh(y_pos, j_counts.values, color=PALETTE["C1"], edgecolor="white", height=0.7)
ax1.set_yticks(y_pos)
ax1.set_yticklabels([j[:40] for j in j_counts.index], fontsize=9)
ax1.invert_yaxis()
ax1.set_xlabel("Articles")
ax1.set_title("a) Top 12 publication venues")
for i, val in enumerate(j_counts.values):
    ax1.text(val + 1, i, str(val), va="center", fontsize=9, fontweight="bold")

# Right: Publisher geography
pub_map = {"arXiv": ("arXiv","US"), "PubMed": ("PubMed","US"),
           "Elsevier": ("Elsevier","NL"), "IEEE": ("IEEE","US"),
           "Springer": ("Springer","DE"), "MDPI": ("MDPI","CH"),
           "Wiley": ("Wiley","US"), "Taylor": ("Taylor&Francis","UK"),
           "SAGE": ("SAGE","US"), "Chapman": ("CRC Press","US"),
           "Qeios": ("Qeios","UK")}
geo = Counter()
for pub in df_filtered["publisher"]:
    for key, (name, country) in pub_map.items():
        if key.lower() in pub.lower():
            geo[f"{name} ({country})"] += 1; break

geo_s = pd.Series(geo).sort_values(ascending=True).tail(10)
country_colors = {"US":"#1565C0","NL":"#2E7D32","DE":"#2E7D32","CH":"#2E7D32","UK":"#E65100"}
colors_g = [country_colors.get(n.split("(")[-1].rstrip(")"), "#888") for n in geo_s.index]
ax2.barh(range(len(geo_s)), geo_s.values, color=colors_g, edgecolor="white", height=0.7)
ax2.set_yticks(range(len(geo_s)))
ax2.set_yticklabels(geo_s.index, fontsize=10)
ax2.set_xlabel("Articles")
ax2.set_title("b) Publisher geography")
for i, val in enumerate(geo_s.values):
    ax2.text(val + 1, i, str(val), va="center", fontsize=9, fontweight="bold")
legend_el = [mpatches.Patch(color="#1565C0", label="USA"),
             mpatches.Patch(color="#2E7D32", label="EU"),
             mpatches.Patch(color="#E65100", label="UK")]
ax2.legend(handles=legend_el, fontsize=9, loc="lower right")

fig.suptitle("Figure 5 — Publication Venue and Publisher Distribution",
             fontsize=15, fontweight="bold", color=PALETTE["navy"], y=1.02)
plt.tight_layout()
savefig(fig, "fig_05")

  FIGURE 5 — Journal & Publisher Distribution
  ✓ fig_05.png


In [17]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FIGURE 6 — CO-AUTHORSHIP NETWORK (Section 3.2)                 ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 6 — Co-Authorship Network")

G = nx.Graph()
edge_w = Counter()
for _, row in df_filtered.iterrows():
    auths = [a.strip() for a in row["authors"].split(";") if a.strip()]
    if len(auths) < 2: continue
    for a1, a2 in combinations(auths[:10], 2):
        edge_w[(a1, a2)] += 1
for (a1, a2), w in edge_w.items():
    G.add_edge(a1, a2, weight=w)

largest_cc = max(nx.connected_components(G), key=len)
sub_nodes = [n for n in largest_cc if G.degree(n) >= 3][:150]
if len(sub_nodes) < 15: sub_nodes = list(largest_cc)[:150]
G_sub = G.subgraph(sub_nodes)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={"width_ratios": [1.4, 1]})

# Left: network
pos = nx.spring_layout(G_sub, k=0.6, iterations=60, seed=42)
degs = dict(G_sub.degree())
node_sz = [min(degs[n] * 18 + 30, 400) for n in G_sub.nodes()]

# Community detection
communities = list(nx.community.label_propagation_communities(G))
communities.sort(key=len, reverse=True)
comm_map = {}
for i, comm in enumerate(communities[:6]):
    for node in comm:
        if node in G_sub: comm_map[node] = i
node_c = [list(PALETTE.values())[comm_map.get(n, 5) % 6] for n in G_sub.nodes()]

nx.draw_networkx_edges(G_sub, pos, alpha=0.12, width=0.6, ax=ax1)
nx.draw_networkx_nodes(G_sub, pos, node_size=node_sz, node_color=node_c, alpha=0.75, ax=ax1)
top_n = sorted(G_sub.nodes(), key=lambda n: degs[n], reverse=True)[:10]
labels = {n: n.split(";")[0][:22] for n in top_n}
nx.draw_networkx_labels(G_sub, pos, labels, font_size=7, font_weight="bold", ax=ax1)
ax1.set_title(f"a) Co-authorship network ({G.number_of_nodes()} authors, {G.number_of_edges()} edges)")
ax1.axis("off")

# Right: degree distribution
degs_all = [d for _, d in G.degree()]
ax2.hist(degs_all, bins=40, color=PALETTE["C1"], edgecolor="white", log=True, zorder=3)
ax2.axvline(np.mean(degs_all), color=PALETTE["red"], ls="--", lw=2,
            label=f"Mean = {np.mean(degs_all):.1f}")
ax2.set_xlabel("Degree (co-authors)")
ax2.set_ylabel("Frequency (log)")
ax2.set_title("b) Degree distribution")
ax2.legend()

fig.suptitle("Figure 6 — Co-Authorship Network Analysis",
             fontsize=15, fontweight="bold", color=PALETTE["navy"], y=1.02)
plt.tight_layout()
savefig(fig, "fig_06")



  FIGURE 6 — Co-Authorship Network
  ✓ fig_06.png


In [18]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FIGURE 7 (a) — KEYWORD CO-OCCURRENCE HEATMAP                        ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 7 (a) — Keyword Co-occurrence Heatmap")

CONCEPTS = [
    "survival analysis", "cox proportional", "kaplan-meier", "deep learning",
    "machine learning", "granger causality", "causal inference", "demand forecasting",
    "time series", "customer churn", "churn prediction", "customer lifetime",
    "random forest", "neural network", "transformer", "hazard ratio",
    "treatment effect", "counterfactual", "bayesian", "time-to-event",
    "competing risk", "supply chain", "energy demand", "regression"
]

def extract_concepts(text):
    text = str(text).lower()
    return {kw for kw in CONCEPTS if kw in text}

concept_freq = Counter()
cooccurrence = Counter()
for _, row in df_filtered.iterrows():
    concepts = extract_concepts(row["full_text"])
    for c in concepts: concept_freq[c] += 1
    for a, b in combinations(sorted(concepts), 2):
        cooccurrence[(a, b)] += 1

top_concepts = [kw for kw, _ in concept_freq.most_common(16)]
cooc_mat = pd.DataFrame(0, index=top_concepts, columns=top_concepts)
for (a, b), cnt in cooccurrence.items():
    if a in top_concepts and b in top_concepts:
        cooc_mat.loc[a, b] = cnt
        cooc_mat.loc[b, a] = cnt

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(cooc_mat, dtype=bool), k=0)
sns.heatmap(cooc_mat, mask=mask, annot=True, fmt="d", cmap="YlOrRd",
            linewidths=0.5, ax=ax, cbar_kws={"label": "Co-occurrence count"},
            xticklabels=[c[:18] for c in top_concepts],
            yticklabels=[c[:18] for c in top_concepts])
ax.set_title("Figure 7 (a) — Keyword Co-occurrence Matrix (Top 16 concepts)",
             fontsize=14, fontweight="bold", color=PALETTE["navy"], pad=15)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
savefig(fig, "fig_07_a")

  FIGURE 7 (a) — Keyword Co-occurrence Heatmap
  ✓ fig_07_a.png


In [19]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FIGURE 7 (b) — KEYWORD CO-OCCURRENCE NETWORK                    ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 7 (b) — Keyword Co-occurrence Network")

def kw_color(kw):
    if kw in ["survival analysis","cox proportional","kaplan-meier","time-to-event",
              "hazard ratio","competing risk"]: return PALETTE["C1"]
    if kw in ["customer churn","churn prediction","customer lifetime"]: return PALETTE["C2"]
    if "granger" in kw: return PALETTE["C3"]
    if kw in ["causal inference","treatment effect","counterfactual"]: return PALETTE["C4"]
    if kw in ["demand forecasting","supply chain","energy demand"]: return PALETTE["C5"]
    return "#888"

G_kw = nx.Graph()
for (a, b), cnt in cooccurrence.items():
    if a in top_concepts and b in top_concepts and cnt >= 3:
        G_kw.add_edge(a, b, weight=cnt)

fig, ax = plt.subplots(figsize=(14, 11))
pos_kw = nx.spring_layout(G_kw, k=2.0, seed=42, weight="weight")
edge_w_kw = [G_kw[u][v]["weight"] / 4 for u, v in G_kw.edges()]
node_sz_kw = [concept_freq.get(n, 10) * 4 for n in G_kw.nodes()]
node_c_kw = [kw_color(n) for n in G_kw.nodes()]

nx.draw_networkx_edges(G_kw, pos_kw, width=edge_w_kw, alpha=0.35, edge_color="#888", ax=ax)
nx.draw_networkx_nodes(G_kw, pos_kw, node_size=node_sz_kw, node_color=node_c_kw, alpha=0.85, ax=ax,
                        edgecolors="white", linewidths=1.5)
nx.draw_networkx_labels(G_kw, pos_kw, font_size=8.5, font_weight="bold", ax=ax)

legend_el = [mpatches.Patch(color=PALETTE[c], label=f"{c}: {CLUSTER_NAMES[c]}") for c in clusters]
ax.legend(handles=legend_el, fontsize=9, loc="upper left", title="Domain")
ax.set_title("Figure (b) — Keyword Co-occurrence Network (threshold ≥ 3)",
             fontsize=14, fontweight="bold", color=PALETTE["navy"])
ax.axis("off")
savefig(fig, "fig_07_b")


  FIGURE 7 (b) — Keyword Co-occurrence Network
  ✓ fig_07_b.png


In [20]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║         FIGURE 8 — MISSING LINKS / GAPS                          ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 8 — Missing Links (Research Gaps)")

expected_links = [
    ("survival analysis", "demand forecasting"),
    ("granger causality", "customer lifetime"),
    ("counterfactual", "demand forecasting"),
    ("competing risk", "customer churn"),
    ("granger causality", "customer churn"),
    ("survival analysis", "supply chain"),
    ("treatment effect", "demand forecasting"),
]
gap_data = []
for a, b in expected_links:
    key = tuple(sorted([a, b]))
    cnt = cooccurrence.get(key, 0)
    gap_data.append({"link": f"{a}  ×  {b}", "count": cnt,
                     "status": "ABSENT" if cnt == 0 else ("WEAK" if cnt < 5 else "Present")})

gap_df = pd.DataFrame(gap_data).sort_values("count", ascending=True)

fig, ax = plt.subplots(figsize=(12, 5.5))
colors_gap = ["#C62828" if s == "ABSENT" else "#F57F17" if s == "WEAK" else "#2E7D32"
              for s in gap_df["status"]]
bars = ax.barh(range(len(gap_df)), gap_df["count"].values, color=colors_gap,
               edgecolor="white", height=0.6)
ax.set_yticks(range(len(gap_df)))
ax.set_yticklabels(gap_df["link"].values, fontsize=10)
ax.set_xlabel("Co-occurrence count")
ax.set_title("Figure 8 — Expected Cross-Domain Links: Identified Gaps",
             fontsize=14, fontweight="bold", color=PALETTE["navy"])
for i, (val, status) in enumerate(zip(gap_df["count"], gap_df["status"])):
    ax.text(max(val + 0.3, 0.3), i, f"[{status}]", va="center", fontsize=9,
            fontweight="bold", color=colors_gap[i])
legend_el = [mpatches.Patch(color="#C62828", label="ABSENT"),
             mpatches.Patch(color="#F57F17", label="WEAK (< 5)"),
             mpatches.Patch(color="#2E7D32", label="Present")]
ax.legend(handles=legend_el, fontsize=9, loc="lower right")
plt.tight_layout()
savefig(fig, "fig_08")


  FIGURE 8 — Missing Links (Research Gaps)
  ✓ fig_08.png


In [21]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║        FIGURE 9 — LDA TOPIC MODELLING                            ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 9 — LDA Topic Modelling")

corpus_texts = []
valid_idx = []
for idx, row in df_filtered.iterrows():
    text = str(row["title_clean"]) + " " + str(row["abstract_clean"])
    if len(text.strip()) > 30:
        corpus_texts.append(text); valid_idx.append(idx)

vectorizer = CountVectorizer(max_df=0.85, min_df=5, max_features=3000,
                              stop_words="english", ngram_range=(1,2),
                              token_pattern=r"[a-zA-Z]{3,}")
dtm = vectorizer.fit_transform(corpus_texts)
feat_names = vectorizer.get_feature_names_out()

N_TOPICS = 6
lda = LatentDirichletAllocation(n_components=N_TOPICS, max_iter=30,
                                 learning_method="online", random_state=42,
                                 doc_topic_prior=0.1, topic_word_prior=0.01)
doc_topics = lda.fit_transform(dtm)

# Extract top words per topic
topic_top_words = {}
for i, topic in enumerate(lda.components_):
    top_idx = topic.argsort()[:-11:-1]
    topic_top_words[f"T{i+1}"] = [(feat_names[j], topic[j]) for j in top_idx]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
topic_colors = [PALETTE["C3"], PALETTE["C2"], PALETTE["C4"], PALETTE["C1"], PALETTE["C5"], PALETTE["C1"]]

for i, ax in enumerate(axes.flat):
    words = topic_top_words[f"T{i+1}"]
    names = [w[0][:20] for w in words]
    weights = [w[1] for w in words]
    max_w = max(weights)
    ax.barh(range(len(names)), weights, color=topic_colors[i % len(topic_colors)],
            edgecolor="white", height=0.7)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(f"Topic {i+1}", fontweight="bold", fontsize=12, color=topic_colors[i % len(topic_colors)])
    ax.set_xlabel("Weight")

fig.suptitle("Figure 9 — LDA Topic Modelling: Top 10 Terms per Topic",
             fontsize=15, fontweight="bold", color=PALETTE["navy"], y=1.02)
plt.tight_layout()
savefig(fig, "fig_09")

  FIGURE 9 — LDA Topic Modelling
  ✓ fig_09.png


In [22]:
df_filtered.head(3)

,doi,title,authors,author_count,year,month,journal,publisher,abstract,subjects,...,abstract_clean,title_clean,full_text,cluster,relevance_score,matched_categories,n_categories,matched_keywords_sample,method_cat,domain_cat
0,10.1002/for.70039,Enhancing Demand Forecasting in Retail: A Comp...,Harsha Chamara Hewage; H. Niles Perera; Kasun ...,3,2026,1.0,Journal of Forecasting,Wiley,<jats:title>ABSTRACT</jats:title>\n ...,,...,ABSTRACT Sales promotions pose challenges to r...,Enhancing Demand Forecasting in Retail: A Comp...,enhancing demand forecasting in retail: a comp...,C5,3,application_domain,1,[T]demand forecasting; [A]demand forecasting,ML/DL,Business
5,10.64898/2025.12.23.25342949,Survival Analysis on Papillary Thyroid Cancer ...,Dilmi Abeywardana; Malinda Iluppangama; Chris ...,3,2026,1.0,,openRxiv,<jats:title>Abstract</jats:title>\n ...,,...,Abstract Cancer accounts for one in six deaths...,Survival Analysis on Papillary Thyroid Cancer ...,survival analysis on papillary thyroid cancer ...,C1,9,method_A_survival,1,[T]survival analysis; [T]cox proportional; [A]...,Survival,Medical
6,10.1063/5.0317814,A novel hybrid method for credit risk predicti...,Fadhaa O. Sameer,1,2026,,AIP Conference Proceedings,AIP Publishing,,,...,,A novel hybrid method for credit risk predicti...,a novel hybrid method for credit risk predicti...,C1,4,method_A_survival,1,[T]cox proportional; [T]proportional hazards,Survival,Other


In [23]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║              FIGURE 10 — TOPIC TEMPORAL TRENDS                   ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 10 — Topic Temporal Trends")

df_valid = df_filtered.loc[valid_idx].copy()
df_valid["topic"] = doc_topics.argmax(axis=1)
topic_year = pd.crosstab(df_valid["year"], df_valid["topic"])
topic_year.index = topic_year.index.astype(int)
topic_year = topic_year.loc[2020:2026]
topic_year.columns = [f"T{c+1}" for c in topic_year.columns]

fig, ax = plt.subplots(figsize=(12, 6))
tc = plt.cm.Set2(np.linspace(0, 1, N_TOPICS))
for i, col in enumerate(sorted(topic_year.columns)):
    ax.plot(topic_year.index, topic_year[col], marker="o", linewidth=2.5,
            color=tc[i], label=col, markersize=7)
ax.set_xlabel("Year"); ax.set_ylabel("Articles")
ax.set_title("Figure 10 — LDA Topic Temporal Evolution",
             fontsize=14, fontweight="bold", color=PALETTE["navy"])
ax.legend(fontsize=9, ncol=2)
ax.set_xticks(topic_year.index)
plt.tight_layout()
savefig(fig, "fig_10")

  FIGURE 10 — Topic Temporal Trends
  ✓ fig_10.png


In [24]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║    FIGURE 11  — t-SNE DOCUMENT MAP                               ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 11 — t-SNE Document Map")

sample_n = min(1500, dtm.shape[0])
rng = np.random.RandomState(42)
sample_idx = rng.choice(dtm.shape[0], sample_n, replace=False)
dtm_s = dtm[sample_idx].toarray()
topics_s = doc_topics[sample_idx].argmax(axis=1)

tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
coords = tsne.fit_transform(dtm_s)

fig, ax = plt.subplots(figsize=(12, 10))
for t in range(N_TOPICS):
    mask = topics_s == t
    ax.scatter(coords[mask, 0], coords[mask, 1], alpha=0.55, s=25,
               label=f"T{t+1}", color=tc[t], edgecolors="white", linewidths=0.3)
ax.set_title("Figure 11 — t-SNE Document Projection by Dominant Topic",
             fontsize=14, fontweight="bold", color=PALETTE["navy"])
ax.legend(fontsize=10, markerscale=2)
ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
plt.tight_layout()
savefig(fig, "fig_11")

  FIGURE 11 — t-SNE Document Map
  ✓ fig_11.png


In [25]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FIGURE 12 TOPIC × CLUSTER HEATMAP                               ║
# ╚══════════════════════════════════════════════════════════════════╝
print(" FIGURE 12 Topic × Cluster Heatmap")

tc_cross = pd.crosstab(df_valid["topic"].apply(lambda x: f"T{x+1}"), df_valid["cluster"])
tc_cross = tc_cross.reindex(columns=["C1","C2","C3","C4","C5","OTHER"], fill_value=0)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(tc_cross, annot=True, fmt="d", cmap="YlGnBu", linewidths=0.5, ax=ax,
            cbar_kws={"label": "Article count"})
ax.set_title("LDA Topics × Thematic Clusters",
             fontsize=14, fontweight="bold", color=PALETTE["navy"])
ax.set_ylabel("LDA Topic"); ax.set_xlabel("Cluster")
plt.tight_layout()
savefig(fig, "fig_12")


 FIGURE 12 Topic × Cluster Heatmap
  ✓ fig_12.png


In [26]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FIGURE 13 - CHI² ASSOCIATION TEST                               ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 13 — Chi² Association Test")

method_order = ["Survival","ML/DL","Causal Inference","Granger","Other"]
domain_order = ["Medical","Business","Energy","Finance","Forecasting","Other"]
contingency = pd.crosstab(df_filtered["method_cat"], df_filtered["domain_cat"])
contingency = contingency.reindex(index=method_order, columns=domain_order, fill_value=0)
chi2, p_val, dof, expected = chi2_contingency(contingency.values)
expected_df = pd.DataFrame(expected, index=contingency.index, columns=contingency.columns)
residuals = (contingency - expected_df) / np.sqrt(expected_df)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

sns.heatmap(contingency, annot=True, fmt="d", cmap="Blues", ax=ax1,
            linewidths=0.5, cbar_kws={"label": "Count"})
ax1.set_title("a) Contingency table", fontsize=13, fontweight="bold")
ax1.set_ylabel("Method"); ax1.set_xlabel("Application domain")

sns.heatmap(residuals, annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=ax2,
            linewidths=0.5, vmin=-8, vmax=8, cbar_kws={"label": "Pearson residual"})
ax2.set_title("b) Standardized Pearson residuals", fontsize=13, fontweight="bold")
ax2.set_ylabel("Method"); ax2.set_xlabel("Application domain")

fig.suptitle(f"Figure — Method × Domain Association Test (χ² = {chi2:.1f}, p < {p_val:.1e})",
             fontsize=15, fontweight="bold", color=PALETTE["navy"], y=1.03)
plt.tight_layout()
savefig(fig, "fig_13")


  FIGURE 13 — Chi² Association Test
  ✓ fig_13.png


In [27]:
contingency.head()

domain_cat,Medical,Business,Energy,Finance,Forecasting,Other
method_cat,,,,,,
Survival,174,14,3,6,9,88
ML/DL,20,40,7,2,12,1
Causal Inference,10,6,2,5,15,40
Granger,2,1,3,5,7,10
Other,4,110,5,3,8,0


In [28]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FIGURE 14 — PERFORMANCE METRICS SYNTHESIS                       ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 14 — Performance Metrics Synthesis")

metric_pats = {
    "C-index": ["c-index","concordance index","c index"],
    "AUC": ["auc","area under","auroc"],
    "Accuracy": ["accuracy"],
    "RMSE": ["rmse","root mean square"],
    "MAPE": ["mape","mean absolute percentage"],
    "F1-score": ["f1-score","f1 score","f-measure"],
    "MAE": ["mae ","mean absolute error"],
    "Brier": ["brier score","brier"],
    "Precision": ["precision"],
    "Recall": ["recall","sensitivity"]
}

metric_counts = defaultdict(lambda: Counter())
metric_total = Counter()
for _, row in df_filtered.iterrows():
    a = row["abstract_clean"].lower()
    for metric, pats in metric_pats.items():
        if any(p in a for p in pats):
            metric_total[metric] += 1
            metric_counts[metric][row["cluster"]] += 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Left: total metric frequency
mt_df = pd.Series(metric_total).sort_values(ascending=True)
ax1.barh(range(len(mt_df)), mt_df.values, color=PALETTE["C1"], edgecolor="white", height=0.65)
ax1.set_yticks(range(len(mt_df)))
ax1.set_yticklabels(mt_df.index, fontsize=11)
ax1.set_xlabel("Articles mentioning metric")
ax1.set_title("a) Metric frequency in abstracts")
for i, val in enumerate(mt_df.values):
    ax1.text(val + 1, i, str(val), va="center", fontsize=10, fontweight="bold")

# Right: metrics by cluster (stacked)
mc = pd.DataFrame(metric_counts).fillna(0).astype(int).T
mc = mc.reindex(columns=["C1","C2","C3","C4","C5","OTHER"], fill_value=0)
mc = mc.loc[mt_df.index]
bottom = np.zeros(len(mc))
for c in ["C1","C2","C3","C4","C5"]:
    if c in mc.columns:
        ax2.barh(range(len(mc)), mc[c].values, left=bottom,
                 label=f"{c}: {CLUSTER_NAMES[c]}", color=PALETTE[c],
                 edgecolor="white", height=0.65)
        bottom += mc[c].values

ax2.set_yticks(range(len(mc)))
ax2.set_yticklabels(mc.index, fontsize=11)
ax2.set_xlabel("Articles")
ax2.set_title("b) Metric usage by cluster")
ax2.legend(fontsize=8, loc="lower right")

fig.suptitle("Figure — Quantitative Performance Metrics Synthesis",
             fontsize=15, fontweight="bold", color=PALETTE["navy"], y=1.02)
plt.tight_layout()
savefig(fig, "fig_14")

  FIGURE 14 — Performance Metrics Synthesis
  ✓ fig_14.png


In [29]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FIGURE — PARADIGM COMPARISON RADAR                              ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 15 — Paradigm Comparison")

# Radar chart for 4 paradigms on 5 dimensions
categories = ["Handles\ncensoring", "Causal\nsemantics", "Temporal\ndynamics",
              "Scalability\n(high-dim)", "Interpretability"]
paradigms = {
    "Survival": [5, 2, 4, 2, 4],
    "Granger": [1, 2, 5, 3, 3],
    "Causal Inf.": [2, 5, 2, 3, 4],
    "ML Forecast": [1, 1, 4, 5, 2]
}
colors_r = [PALETTE["C1"], PALETTE["C3"], PALETTE["C4"], PALETTE["C5"]]

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_rlabel_position(0)
plt.yticks([1,2,3,4,5], ["1","2","3","4","5"], color="grey", size=8)
plt.ylim(0, 5.5)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10, fontweight="bold")

for i, (name, values) in enumerate(paradigms.items()):
    vals = values + values[:1]
    ax.plot(angles, vals, linewidth=2.5, label=name, color=colors_r[i])
    ax.fill(angles, vals, alpha=0.08, color=colors_r[i])

ax.legend(loc="lower right", bbox_to_anchor=(1.3, 0), fontsize=10)
ax.set_title("Paradigm Capability Comparison",
             fontsize=14, fontweight="bold", color=PALETTE["navy"], pad=25)
savefig(fig, "fig_15")


  FIGURE 15 — Paradigm Comparison
  ✓ fig_15.png


In [30]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FIGURE 16 — CLUSTER OVERLAP MATRIX                              ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 16 — Cluster Overlap Matrix")

cluster_sets = defaultdict(set)
for idx, row in df_filtered.iterrows():
    t = str(row["full_text"]).lower()
    for cn, kws in {"C1":["survival","cox","hazard","censored","kaplan"],
                     "C2":["churn","customer lifetime","retention","subscriber"],
                     "C3":["granger causality","granger cause"],
                     "C4":["causal inference","counterfactual","treatment effect"],
                     "C5":["demand forecasting","demand prediction","energy demand","supply chain"]}.items():
        if any(kw in t for kw in kws):
            cluster_sets[cn].add(idx)

overlap = pd.DataFrame(0, index=clusters, columns=clusters)
for c in clusters:
    overlap.loc[c, c] = len(cluster_sets[c])
for c1, c2 in combinations(clusters, 2):
    ov = len(cluster_sets[c1] & cluster_sets[c2])
    overlap.loc[c1, c2] = ov
    overlap.loc[c2, c1] = ov

fig, ax = plt.subplots(figsize=(9, 8))
labels_ov = [f"{c}\n{CLUSTER_NAMES[c]}" for c in clusters]
mask_diag = np.eye(len(clusters), dtype=bool)
sns.heatmap(overlap.values, annot=True, fmt="d", cmap="YlOrRd",
            xticklabels=labels_ov, yticklabels=labels_ov, ax=ax,
            linewidths=0.5, cbar_kws={"label": "Articles"})
ax.set_title("Cluster Overlap Matrix (multi-domain articles)",
             fontsize=14, fontweight="bold", color=PALETTE["navy"], pad=15)
plt.tight_layout()
savefig(fig, "fig_16")


  FIGURE 16 — Cluster Overlap Matrix
  ✓ fig_16.png


In [31]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FIGURE 17 — INTEGRATION GAP ANALYSIS                            ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 17 — Integration Gap Analysis")

intersections = {
    "S only": len(cluster_sets["C1"] - cluster_sets["C3"] - cluster_sets["C4"] - cluster_sets["C5"]),
    "G only": len(cluster_sets["C3"] - cluster_sets["C1"] - cluster_sets["C4"] - cluster_sets["C5"]),
    "C only": len(cluster_sets["C4"] - cluster_sets["C1"] - cluster_sets["C3"] - cluster_sets["C5"]),
    "D only": len(cluster_sets["C5"] - cluster_sets["C1"] - cluster_sets["C3"] - cluster_sets["C4"]),
    "S∩G": len(cluster_sets["C1"] & cluster_sets["C3"]),
    "S∩C": len(cluster_sets["C1"] & cluster_sets["C4"]),
    "S∩D": len(cluster_sets["C1"] & cluster_sets["C5"]),
    "G∩C": len(cluster_sets["C3"] & cluster_sets["C4"]),
    "G∩D": len(cluster_sets["C3"] & cluster_sets["C5"]),
    "C∩D": len(cluster_sets["C4"] & cluster_sets["C5"]),
    "S∩G∩C": len(cluster_sets["C1"] & cluster_sets["C3"] & cluster_sets["C4"]),
    "S∩G∩D": len(cluster_sets["C1"] & cluster_sets["C3"] & cluster_sets["C5"]),
    "S∩C∩D": len(cluster_sets["C1"] & cluster_sets["C4"] & cluster_sets["C5"]),
    "G∩C∩D": len(cluster_sets["C3"] & cluster_sets["C4"] & cluster_sets["C5"]),
    "S∩G∩C∩D": len(cluster_sets["C1"] & cluster_sets["C3"] & cluster_sets["C4"] & cluster_sets["C5"]),
}

int_df = pd.DataFrame(list(intersections.items()), columns=["Set", "N"])
int_df = int_df.sort_values("N", ascending=True)

fig, ax = plt.subplots(figsize=(13, 7))
colors_int = []
for name in int_df["Set"]:
    if "∩" not in name: colors_int.append("#90CAF9")
    elif name.count("∩") == 1: colors_int.append(PALETTE["amber"])
    elif name.count("∩") == 2: colors_int.append("#E53935")
    else: colors_int.append("#B71C1C")

ax.barh(range(len(int_df)), int_df["N"].values, color=colors_int, edgecolor="white", height=0.65)
ax.set_yticks(range(len(int_df)))
ax.set_yticklabels(int_df["Set"].values, fontsize=10, fontweight="bold")
ax.set_xlabel("Number of articles")
for i, val in enumerate(int_df["N"].values):
    ax.text(val + 1, i, str(val), va="center", fontsize=10, fontweight="bold",
            color=colors_int[i])

legend_el = [mpatches.Patch(color="#90CAF9", label="Single domain"),
             mpatches.Patch(color=PALETTE["amber"], label="2-way intersection"),
             mpatches.Patch(color="#E53935", label="3-way intersection"),
             mpatches.Patch(color="#B71C1C", label="4-way (full integration)")]
ax.legend(handles=legend_el, fontsize=9, loc="lower right")
ax.set_title("Integration Gap: Domain Intersections (S=Survival, G=Granger, C=Causal, D=Demand)",
             fontsize=13, fontweight="bold", color=PALETTE["navy"])
plt.tight_layout()
savefig(fig, "fig_17")


  FIGURE 17 — Integration Gap Analysis
  ✓ fig_17.png


In [32]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FIGURE 18 — CONTRADICTIONS MATRIX                               ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 18 — Contradictions Visual")

contradictions = [
    ("Cox PH\nrobust?", "Austin & Giardiello\n(2025)", "Post et al.\n(2024)", "Fundamental"),
    ("DL > Cox?", "Wiegrebe et al.\n(2024)", "Rossi et al.\n(2025)", "Major"),
    ("Granger\ndecidable?", "Bloniasz\n(2022)", "Lin et al.\n(2025)", "Major"),
    ("HR causal?", "Ji & Oganisian\n(2023)", "Post et al.\n(2024)", "Fundamental"),
    ("Prediction\n≠ causation?", "Bastías & Brand\n(2020)", "Kolassa et al.\n(2023)", "Epistemological"),
    ("Survival\n> binary?", "Dananjaya\n(2026)", "Phetvilai\n(2025)", "Practical"),
]

fig, ax = plt.subplots(figsize=(16, 8))
ax.set_xlim(0, 16); ax.set_ylim(-0.5, len(contradictions) - 0.5)
ax.axis("off")

sev_colors = {"Fundamental": "#C62828", "Major": "#E65100", "Epistemological": "#7B1FA2", "Practical": "#1565C0"}

for i, (subject, artA, artB, severity) in enumerate(contradictions):
    y = len(contradictions) - 1 - i
    # Subject
    ax.text(0.5, y, f"#{i+1}", fontsize=12, fontweight="bold", va="center",
            color=sev_colors[severity])
    ax.text(2.2, y, subject, fontsize=10, fontweight="bold", va="center",
            multialignment="center", color=PALETTE["text"])
    # Article A (blue box)
    rect_a = FancyBboxPatch((4, y - 0.35), 4.5, 0.7, boxstyle="round,pad=0.1",
                             facecolor="#E3F2FD", edgecolor="#1565C0", linewidth=1.5)
    ax.add_patch(rect_a)
    ax.text(6.25, y, artA, fontsize=9, va="center", ha="center",
            multialignment="center", color="#1565C0")
    # VS
    ax.text(9, y, "vs", fontsize=11, fontweight="bold", va="center", ha="center",
            color="#999", style="italic")
    # Article B (amber box)
    rect_b = FancyBboxPatch((9.5, y - 0.35), 4.5, 0.7, boxstyle="round,pad=0.1",
                             facecolor="#FFF8E1", edgecolor="#F57F17", linewidth=1.5)
    ax.add_patch(rect_b)
    ax.text(11.75, y, artB, fontsize=9, va="center", ha="center",
            multialignment="center", color="#E65100")
    # Severity badge
    ax.text(15, y, severity, fontsize=8.5, fontweight="bold", va="center",
            ha="center", color="white",
            bbox=dict(boxstyle="round,pad=0.2", facecolor=sev_colors[severity]))

ax.set_title("Figure — Six Major Contradictions Identified in the Literature",
             fontsize=14, fontweight="bold", color=PALETTE["navy"], pad=20)
savefig(fig, "fig_18")


  FIGURE 18 — Contradictions Visual
  ✓ fig_18.png


In [33]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FIGURE 19 — RANKED RESEARCH GAPS                                ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE 19 — Ranked Research Gaps")

gaps = [
    ("CRITICAL", "No integrated Survival×Granger×Causal\n×Demand pipeline exists", 1.0),
    ("MAJOR", "Demand lifecycle modelling as\ntime-to-event process unexplored", 0.8),
    ("MAJOR", "Granger not used for causal\nfeature selection in survival/forecasting", 0.8),
    ("MODERATE", "Counterfactual demand estimation\nunder interventions is rare", 0.6),
    ("MODERATE", "Unified evaluation metrics across\nsurvival/causal/forecasting absent", 0.6),
]

fig, ax = plt.subplots(figsize=(13, 6))
gap_colors_map = {"CRITICAL": "#C62828", "MAJOR": "#E65100", "MODERATE": "#F57F17"}
for i, (priority, label, width) in enumerate(gaps):
    y = len(gaps) - 1 - i
    color = gap_colors_map[priority]
    ax.barh(y, width, color=color, edgecolor="white", height=0.55, zorder=3)
    ax.text(width + 0.03, y, f"[{priority}]", va="center", fontsize=10,
            fontweight="bold", color=color)
    ax.text(-0.02, y, label, va="center", ha="right", fontsize=10, color=PALETTE["text"],
            multialignment="right")

ax.set_xlim(-0.02, 1.25)
ax.set_yticks([])
ax.set_xlabel("Priority (relative)")
ax.set_title("Figure — Ranked Research Gaps",
             fontsize=14, fontweight="bold", color=PALETTE["navy"])
ax.axvline(0, color="#444", linewidth=1)
plt.tight_layout()
savefig(fig, "fig_19")

  FIGURE 19 — Ranked Research Gaps
  ✓ fig_19.png


In [34]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FIGURE 20 — CSDF FRAMEWORK ARCHITECTURE                         ║
# ╚══════════════════════════════════════════════════════════════════╝
print("  FIGURE  20 — CSDF Framework Architecture")

fig, ax = plt.subplots(figsize=(18, 13))
ax.set_xlim(0, 18); ax.set_ylim(0, 14)
ax.axis("off")

# Title
ax.text(9, 13.3, "CAUSAL-SURVIVAL DEMAND FORECASTING (CSDF) FRAMEWORK",
        fontsize=16, fontweight="bold", ha="center", color=PALETTE["navy"])
ax.text(9, 12.9, "Proposed Multi-Layer Integration Architecture",
        fontsize=11, ha="center", color="#666", style="italic")

# Layers
layers = [
    (9, 11.5, "LAYER 1 — Granger-Based Causal Feature Selection",
     "KAN-Granger  ·  LASSO-VAR  ·  Spectral Granger\n"
     "Input: Multivariate time series  →  Output: Ranked causal features + lag structure",
     "#E3F2FD", "#1565C0"),
    (9, 9.2, "LAYER 2 — Survival-Based Demand Lifecycle Modelling",
     "Cox PH  ·  DeepHit  ·  SurvKAN  ·  Competing Risks\n"
     "Input: Individual histories + L1 features  →  Output: Hazard functions, survival curves",
     "#E8F5E9", "#2E7D32"),
    (9, 6.9, "LAYER 3 — Structural Causal Counterfactual Engine",
     "TMLE  ·  DML  ·  Orthogonal Survival Learners\n"
     "Input: DAG (from L1+L2) + interventions  →  Output: ATE/CATE, counterfactual trajectories",
     "#F3E5F5", "#6A1B9A"),
    (9, 4.6, "LAYER 4 — Integrative Forecast Aggregation",
     "Meta-learner (stacking/BMA)  ·  Uncertainty quantification\n"
     "Input: L1+L2+L3 + traditional forecasts  →  Output: Calibrated causal demand forecasts",
     "#FFEBEE", "#C62828"),
]

for x, y, title, desc, fc, ec in layers:
    rect = FancyBboxPatch((x - 6.5, y - 0.8), 13, 1.6, boxstyle="round,pad=0.2",
                           facecolor=fc, edgecolor=ec, linewidth=2.5, alpha=0.95)
    ax.add_patch(rect)
    ax.text(x, y + 0.35, title, ha="center", va="center", fontsize=12,
            fontweight="bold", color=ec)
    ax.text(x, y - 0.3, desc, ha="center", va="center", fontsize=10,
            color="#444", multialignment="center", family="monospace")

# Arrows between layers
for i in range(len(layers) - 1):
    y_start = layers[i][1] - 0.85
    y_end = layers[i+1][1] + 0.85
    ax.annotate("", xy=(9, y_end), xytext=(9, y_start),
                arrowprops=dict(arrowstyle="-|>", color="#333", lw=3, mutation_scale=22))

# Side annotations (resolves which incompatibility)
side_notes = [
    (16, 11.5, "Resolves:\nIncompat. (i)\nData granularity"),
    (16, 9.2, "Resolves:\nIncompat. (ii)\nStationarity vs censoring"),
    (16, 6.9, "Resolves:\nIncompat. (iii)\nCausal semantics gap"),
    (16, 4.6, "Resolves:\nGap #5\nUnified metrics"),
]
for x, y, text in side_notes:
    ax.text(x, y, text, ha="center", va="center", fontsize=8.5, color="#555",
            multialignment="center",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="#FAFAFA", edgecolor="#CCC"))
    ax.annotate("", xy=(15.5, y), xytext=(14.5, y),
                arrowprops=dict(arrowstyle="<-", color="#CCC", lw=1.5))

# Use cases at bottom
ax.text(9, 2.8, "PRACTICAL USE CASES", fontsize=12, fontweight="bold",
        ha="center", color=PALETTE["navy"])
use_cases = [
    ("Retail", "Demand forecasting\nunder promotions", PALETTE["C5"]),
    ("Telecom", "Churn management\n& retention", PALETTE["C2"]),
    ("Energy", "Grid planning &\nload forecasting", PALETTE["C3"]),
]
for i, (name, desc, color) in enumerate(use_cases):
    cx = 3.5 + i * 5.5
    rect = FancyBboxPatch((cx - 2, 1.2), 4, 1.2, boxstyle="round,pad=0.15",
                           facecolor=color, edgecolor="white", linewidth=2, alpha=0.15)
    ax.add_patch(rect)
    ax.text(cx, 2.0, name, ha="center", va="center", fontsize=11,
            fontweight="bold", color=color)
    ax.text(cx, 1.5, desc, ha="center", va="center", fontsize=9,
            color="#444", multialignment="center")

# Arrow from L4 to use cases
ax.annotate("", xy=(9, 3.1), xytext=(9, 3.75),
            arrowprops=dict(arrowstyle="-|>", color="#333", lw=2, mutation_scale=18))

savefig(fig, "fig_20")

  FIGURE  20 — CSDF Framework Architecture
  ✓ fig_20.png


In [35]:
# ═══════════════════════════════════════════════════════════════════
#  SUMMARY
# ═══════════════════════════════════════════════════════════════════
figs = sorted(os.listdir(OUT))
print(f"\n{'═'*60}")
print(f"  ✅ {len(figs)} FIGURES GENERATED SUCCESSFULLY")
print(f"{'═'*60}")
for f in figs:
    size = os.path.getsize(f"{OUT}/{f}") / 1024
    print(f"  {f:.<50} {size:.0f} KB")
print(f"\n  Output: {OUT}/")



════════════════════════════════════════════════════════════
  ✅ 28 FIGURES GENERATED SUCCESSFULLY
════════════════════════════════════════════════════════════
  .DS_Store......................................... 6 KB
  .ipynb_checkpoints................................ 1 KB
  Backup............................................ 1 KB
  combined_bibliometric_data.csv.................... 2110 KB
  fig_01.png........................................ 106 KB
  fig_02.png........................................ 142 KB
  fig_03.png........................................ 147 KB
  fig_04.png........................................ 202 KB
  fig_05.png........................................ 194 KB
  fig_06.png........................................ 294 KB
  fig_06_bis.png.................................... 408 KB
  fig_07.png........................................ 136 KB
  fig_07_a.png...................................... 313 KB
  fig_07_b.png...................................... 409 KB
  fi